In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

## Install libraries

```bash
uv sync --upgrade
```

## Update repository

In [ ]:
# ! git pull

## Add import path

In [ ]:
import gc
import os
import sys
from pathlib import Path

In [ ]:
def add_library_level(level=4):
    suf_path = ['..']
    path = '..'
    for i in range(0, level):
        join_path = suf_path * i
        path = '/'.join(join_path)
        module_path = os.path.abspath(os.path.join(path))
        if module_path not in sys.path:
            sys.path.append(module_path)
            print(f'Appendeding {path}')

In [ ]:
add_library_level(level=5)

## Organize imports

In [ ]:
import multiprocessing

In [ ]:
from src.lattmc.utils import init_device, empty_cache
from src.lattmc.paths import tr_paths, sae_paths

In [ ]:
from src.lattmc.fca.utils import *
from src.lattmc.fca.lattice_utils import diff, subset
from src.lattmc.fca.fca_utils import *
from src.lattmc.fca.info_utils import FCAtoPCA
from src.lattmc.latex.latex_utils import print_latex_text

In [ ]:
from src.lattmc.tc.tokenization_utils import BG_CODES

In [ ]:
from src.lattmc.tc.transcoder_analyzers import ConceptAnalysis, init_analyzer

In [ ]:
from src.lattmc.sae.nlp_sae_utils import gen_concept

In [ ]:
import logging

#### Number of CPU cores

In [ ]:
workers = multiprocessing.cpu_count()
workers

In [ ]:
SEED = 2026

In [ ]:
logging.basicConfig(level=logging.INFO)

In [ ]:
device = init_device()
device

In [ ]:
torch.__version__, np.__version__

## Initialize Paths

In [ ]:
GPT2, TOKENS_PATH = tr_paths()

In [ ]:
! ls {GPT2}

In [ ]:
GPT2.absolute()

In [ ]:
! ls {TOKENS_PATH}

In [ ]:
! ls {str(TOKENS_PATH.absolute())}

## Load trancoders

In [ ]:
layers = list(range(12))
# layers = [0, 4, 6, 8, 10, 11]
# layers = [0, 5, 7, 9, 11]
layers = [0, 7, 8]

In [ ]:
tr_analyzer = init_analyzer(
    layers,
    TOKENS_PATH,
    GPT2,
    model_name='gpt2-small',
    device=device,
    tr_or_sae=True,
    vector_dir=GPT2,
)

In [ ]:
empty_cache()

In [ ]:
with tqdm(layers) as players:
    nz_fcas = {layer: NonzeroFCA.fromFCA(tr_analyzer.fcas[layer]) for layer in players}
    nz_analyzer = tr_analyzer.fromAnalyzer(
        tr_analyzer,
        fcas = nz_fcas,
    )

In [ ]:
empty_cache()

In [ ]:
tr_analyzer.transcoder.model.cfg.device

In [ ]:
empty_cache()

## Features Dimension

In [ ]:
tr_analyzer.fcas[layers[0]].V.shape, tr_analyzer.fcas[layers[1]].V.shape, tr_analyzer.fcas[layers[-1]].V.shape, tr_analyzer.tokens.shape

## Select Sample $A \in X$

In [ ]:
# A_nyc = np.random.randint(0, high=tr_analyzer.corpus.shape[0], size=11)
# A_nyc

In [ ]:
# A_nyc = np.array([
#     25576, 
#     19744,  
#     4488,  
#     8217, 
#     21048, 
#     11717, 
#     15774,  
#     1431, 
#     19151,
#     15355, 
#     24614
# ])

In [ ]:
# A_nyc = np.array([17525, 11882,  1305,  2024, 19824,  7857,  5753, 20095, 22451,
#          841, 10946])

In [ ]:
# A_nyc = np.array([11882, 1305, 7857, 20095, 22451, 841, 10946])

In [ ]:
# A_nyc = np.array([1305, 7857, 22451, 841, 10946])

In [ ]:
A_nyc = np.array([25314,  7705, 25413,  1757,  9276, 19655, 23050, 18379, 19094,
        8099, 17518, 13766, 20164,  2949, 15098, 11772, 23708])

## Mapping on layer 0

In [ ]:
layer = 0

#### Calculate $G_{I}F_{I}$

In [ ]:
conA = tr_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(conA.v)
val, idcs, val.shape

In [ ]:
A_nyc.shape, conA.A.shape

In [ ]:
_, idxs = topNonZeros(conA.v)
max_idx = idxs.shape[0]
max_idx

In [ ]:
p_k = topRangeProject(conA.v, from_idx=10, to_idx=25)
topK(p_k, k=5)

In [ ]:
p_test = topRangeProject(conA.v, from_idx=0, to_idx=max_idx)
topK(p_test, k=7), p_test.shape

In [ ]:
A_sub = np.array([
    25314,
    7705,
    25413,
    1757,
    9276
])

In [ ]:
A_nyc[:5]

In [ ]:
_ = tr_analyzer.print_all_from_objects(
    layer,
    A_nyc[:5],
    p_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
multitxt_l0 = tr_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    p_test,
    limit=5,
)

In [ ]:
print_latex_text(multitxt_l0)

In [ ]:
pres = tr_analyzer.print_all_from_objects(
    layer,
    conA.A,
    p_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
_ = tr_analyzer.print_multitexts_from_objects(
    layer,
    # conA.A,
    A_nyc[:5],
    p_test,
    limit=5,
)

### Minimal FCA $G_{J}F_{J}$ and Analysis

In [ ]:
nozA = nz_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(nozA.v)
val, idcs, val.shape

In [ ]:
A_nyc.shape, nozA.A.shape

In [ ]:
n_k = topRangeProject(nozA.v, from_idx=10, to_idx=25)
topK(n_k, k=5)

In [ ]:
n_test = topRangeProject(nozA.v, from_idx=0, to_idx=max_idx)
topK(n_test, k=5), n_test.shape

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    A_nyc,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
multitxt_z0 = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
print_latex_text(multitxt_z0)

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    nozA.A,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
_ = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
intA = intersect(conA.A, nozA.A)
np.all(intA == conA.A)

## Mapping on layer 8

In [ ]:
layer = 8

#### Calculate $G_{I}F_{I}$

In [ ]:
conA = tr_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(conA.v)
val, idcs, val.shape

In [ ]:
A_nyc.shape, conA.A.shape

In [ ]:
_, idxs = topNonZeros(conA.v)
max_idx = idxs.shape[0]
max_idx

In [ ]:
p_k = topRangeProject(conA.v, from_idx=10, to_idx=25)
topK(p_k, k=5)

In [ ]:
p_test = topRangeProject(conA.v, from_idx=0, to_idx=max_idx)
topK(p_test, k=7), p_test.shape

In [ ]:
pres = tr_analyzer.print_all_from_objects(
    layer,
    A_nyc,
    p_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
multitxt_l8 = tr_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    p_test,
    limit=5,
)

In [ ]:
print_latex_text(multitxt_l8)

In [ ]:
pres = tr_analyzer.print_all_from_objects(
    layer,
    conA.A,
    p_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
_ = tr_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    p_test,
    limit=5,
)

### Minimal FCA $G_{J}F_{J}$ and Analysis

In [ ]:
nozA = nz_analyzer.fcas[layer].GF_F(A_nyc)
val, idcs = topNonZeros(nozA.v)
val, idcs, val.shape

In [ ]:
A_nyc.shape, nozA.A.shape

In [ ]:
n_k = topRangeProject(nozA.v, from_idx=10, to_idx=25)
topK(n_k, k=5)

In [ ]:
n_test = topRangeProject(nozA.v, from_idx=0, to_idx=max_idx)
topK(n_test, k=5), n_test.shape

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    A_nyc,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
multitxt_z8 = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
print_latex_text(multitxt_z8)

In [ ]:
pres = nz_analyzer.print_all_from_objects(
    layer,
    nozA.A,
    n_k,
    with_text=True,
    limit=5,
    log_tokens_and_idxs=False,
)

In [ ]:
_ = nz_analyzer.print_multitexts_from_objects(
    layer,
    A_nyc[:5],
    n_test,
    limit=5,
)

In [ ]:
intA = intersect(conA.A, nozA.A)
np.all(intA == conA.A)

## Experiments on Token FCA

In [ ]:
tr_analyzer.tokenize('New').to('cpu')[0][-1]

In [ ]:
T = [
    tr_analyzer.tokenize(' New').to('cpu').tolist()[0][-1],
    tr_analyzer.tokenize(' York').to('cpu').tolist()[0][-1],
    tr_analyzer.tokenize(' City').to('cpu').tolist()[0][-1],
    # tr_analyzer.tokenize(' Official').to('cpu').tolist()[0][-1],
]
T

In [ ]:
S_ny = {968, 1971}
S_ny

In [ ]:
tr_analyzer.transcoder.print_tokens_with_text(T)

In [ ]:
T

### Initialize FCA for tokens

In [ ]:
pca = TokenFCA(
    tr_analyzer.corpus,
    T
)

In [ ]:
S = set(T)
S, T[0], T[1], is_empty(S)

In [ ]:
G_S = pca.G(S)
G_S

In [ ]:
F_S = pca.F(G_S)
F_S, S

In [ ]:
powS = powerset_bit(S)
powS, pca.powS

In [ ]:
cl_arr = pca.are_attributes_closed()
cl_arr

In [ ]:
G_S1 = pca.G({T[0]})
G_S2 = pca.G({T[1]})
G_S12 = intersect(G_S1, G_S2)
G_S = pca.G(S)

In [ ]:
G_S1.shape, G_S2.shape, G_S12.shape, G_S.shape, np.array_equal(G_S, G_S12) 

In [ ]:
_ = tr_analyzer.match_tokens_in_prompts_and_print(G_S[:5], S)

In [ ]:
G_S_ny = pca.G(S_ny)

In [ ]:
S_ny

In [ ]:
G_S_diff = diff(G_S_ny, G_S)

In [ ]:
text_dict = tr_analyzer.match_tokens_in_prompts_and_print(G_S_diff[:5], S_ny, color_idx=35)

In [ ]:
print(print_latex_text(text_dict))

In [ ]:
S_ny = set(T[:2])
S_ny, tr_analyzer.to_clean(T[:2])

In [ ]:
G_Ts = list()
for t in S_ny:
    G_t = pca.G(set({t}))
    G_Ts.append(G_t)

## Init Infomorphisms

In [ ]:
infoms = {layer: FCAtoPCA(
    tr_analyzer.fcas[layer],
    pca
) for layer in layers}

### Map on Layer 0

In [ ]:
layer = 0

In [ ]:
infom = infoms[layer]

In [ ]:
infoms

In [ ]:
infom.g_st

In [ ]:
infom.powS

In [ ]:
G_S_ny = pca.G(S_ny)
G_S_ny.shape

In [ ]:
infom.g(S_ny)

In [ ]:
A_ny = G_S_ny[:9]
A_ny

In [ ]:
infom.powS[[0, 2]]

In [ ]:
f_A = infom.f(A_ny)
f_A.shape

In [ ]:
subset(A_ny, f_A)

In [ ]:
intersect(A_ny, f_A)

In [ ]:
G_I_F_I_A_ny = infom.fca.GF(A_ny)
f_G_I_F_I_A_ny = infom.f(G_I_F_I_A_ny)
f_A_ny = infom.f(A_ny)
G_M_F_M_f_A_ny = infom.pca.GF(f_A_ny)

In [ ]:
f_G_I_F_I_A_ny.shape, G_M_F_M_f_A_ny.shape

In [ ]:
subset(f_G_I_F_I_A_ny, G_M_F_M_f_A_ny)

In [ ]:
gS = infom.g(S_ny)
topNonZeros(gS)

In [ ]:
infom.fca.G(gS)

In [ ]:
pre_S_f = infom.pre_f(A_ny)
pre_S_f

In [ ]:
f_A

In [ ]:
text_dict = tr_analyzer.match_tokens_in_prompts_and_print(f_A[:5], pre_S_f, color_idx=35)

In [ ]:
print_latex_text(text_dict)

In [ ]:
f_A_sam = [key.item() for key in text_dict.keys()]
f_A_sam

In [ ]:
f_A_stat = [11136, 3457, 22529, 20356, 11014]

### Map on Layer 8

In [ ]:
layer = 7

In [ ]:
infom = infoms[layer]

In [ ]:
infoms

In [ ]:
infom.g_st

In [ ]:
infom.powS

In [ ]:
G_S_ny = pca.G(S_ny)
G_S_ny.shape

In [ ]:
infom.g(S_ny)

In [ ]:
A_ny = G_S_ny[:9]
A_ny

In [ ]:
infom.powS[[0, 2]]

In [ ]:
f_A = infom.f(A_ny)
f_A.shape

In [ ]:
subset(A_ny, f_A)

In [ ]:
intersect(A_ny, f_A)

In [ ]:
G_I_F_I_A_ny = infom.fca.GF(A_ny)
f_G_I_F_I_A_ny = infom.f(G_I_F_I_A_ny)
f_A_ny = infom.f(A_ny)
G_M_F_M_f_A_ny = infom.pca.GF(f_A_ny)

In [ ]:
f_G_I_F_I_A_ny.shape, G_M_F_M_f_A_ny.shape

In [ ]:
subset(f_G_I_F_I_A_ny, G_M_F_M_f_A_ny)

In [ ]:
gS = infom.g(S_ny)
topNonZeros(gS)

In [ ]:
infom.fca.G(gS)

In [ ]:
pre_S_f = infom.pre_f(A_ny)
pre_S_f

In [ ]:
f_A

In [ ]:
text_dict = tr_analyzer.match_tokens_in_prompts_and_print(f_A[:5], pre_S_f, color_idx=35)

In [ ]:
print_latex_text(text_dict)